# Assignment 2 — Transformer Language Model

This assignment extends Assignment 1 by replacing the LSTM with a **Transformer** (OLMo 2 architecture)
and adding text generation. The pipeline:

1. **Transformer components** — SwiGLU MLP, RMSNorm, RoPE, Multi-Head Attention, Decoder layer, full stack
2. **Training** — reuse `A1Trainer` from assignment 1
3. **Text generation** — greedy next-word prediction, temperature/top-K sampling, OLMo-2 comparison

All `nn.Linear` layers use `bias=False`. `hidden_size` must be divisible by `num_attention_heads`.

In [1]:
import os, sys
import torch
from torch import nn
from torch.distributions import Categorical
from transformers import PreTrainedModel, PretrainedConfig, TrainingArguments
from transformers.modeling_outputs import CausalLMOutput

# Reuse tokenizer, dataset loader and trainer from Assignment 1
sys.path.insert(0, '../../ass_1_tokenisation_and_embeddings/a1_1')
from A1_skeleton import load_tokenizer, get_dataset, A1Trainer, A1Tokenizer, get_i2t

/Users/matheus/dev/phd/dl4nlp/wasp-deep-learning-for-nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Part 1 — Transformer Components

### Model configuration

All hyperparameters are stored in a `PretrainedConfig` subclass so the model can be saved and reloaded
with `save_pretrained` / `from_pretrained`.

In [ ]:
class A2ModelConfig(PretrainedConfig):
    """Hyperparameters for the Transformer language model."""
    def __init__(self, vocab_size=None, hidden_size=None, intermediate_size=None,
                 num_attention_heads=None, num_hidden_layers=None, rope_theta=None,
                 hidden_act='silu', max_position_embeddings=None, rms_norm_eps=None, **kwargs):
        super().__init__(**kwargs)
        self.loss_type             = 'ForCausalLM'
        self.vocab_size            = vocab_size
        self.hidden_size           = hidden_size
        self.intermediate_size     = intermediate_size
        self.num_attention_heads   = num_attention_heads
        self.num_hidden_layers     = num_hidden_layers
        self.rope_theta            = rope_theta
        self.hidden_act            = hidden_act
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps          = rms_norm_eps


# Shared small config used for all sanity checks below
TEST_CONFIG = A2ModelConfig(
    vocab_size=10000, 
    hidden_size=6, 
    intermediate_size=11,
    num_attention_heads=3, 
    num_hidden_layers=2,
    rope_theta=100000.0, 
    hidden_act='silu',
    max_position_embeddings=512, 
    rms_norm_eps=1e-6,
)
# Dummy 3-D tensor: (B=4, N=3, H=6)
DUMMY = torch.ones(4, 3, 6, dtype=torch.float32)

random_tensor = torch.tensor( # the first dimension has size 4, the second has size 3, the last has size 6
    [
        [
            [1,1,1,1,1,1], [1,1,1,1,1,1], [1,1,1,1,1,1]
        ],
        [
            [1,1,1,1,1,1], [1,1,1,1,1,1], [1,1,1,1,1,1]
        ],
        [
            [1,1,1,1,1,1], [1,1,1,1,1,1], [1,1,1,1,1,1]
        ],
        [
            [1,1,1,1,1,1], [1,1,1,1,1,1], [1,1,1,1,1,1]
        ],
    ], dtype=torch.float32)

### Task 1.1 — SwiGLU MLP 🎓

The MLP uses the **SwiGLU** activation (used in LLaMA, OLMo, …). Instead of a single projection followed
by an activation, it computes *two* parallel projections and multiplies them element-wise:

$$\text{MLP}(x) = W_{\text{top}}\,\bigl(W_{\text{right}}(x) \odot \text{SiLU}(W_{\text{left}}(x))\bigr)$$

The ⊗ symbol in the assignment diagram refers to this element-wise multiplication.

Shapes: 

- input `(B, N, H)` → intermediate `(B, N, I)` → output `(B, N, H)`  

where `H` = `hidden_size` and `I` = `intermediate_size`.

<img src="../../images/swiglu.svg" width="150" style="background-color: white; padding: 8px;"/>


In [4]:
class A2MLP(nn.Module):
    """SwiGLU feed-forward block."""
    def __init__(self, config):
        super().__init__()
        assert config.hidden_act == 'silu'
        self.linear_right = nn.Linear(in_features=config.hidden_size,       out_features=config.intermediate_size, bias=False)
        self.linear_left  = nn.Linear(in_features=config.hidden_size,       out_features=config.intermediate_size, bias=False)
        self.linear_top   = nn.Linear(in_features=config.intermediate_size, out_features=config.hidden_size, bias=False)
        self.silu = nn.SiLU()

    def forward(self, hidden_states):
        gate   = self.silu(self.linear_left(hidden_states))   # SiLU gate
        hidden = self.linear_right(hidden_states) * gate      # element-wise gating
        return self.linear_top(hidden)

In [5]:
# Sanity check: output shape should equal input shape (4, 3, 6)
mlp = A2MLP(TEST_CONFIG)
out = mlp(DUMMY)
assert out.shape == DUMMY.shape, f'Expected {DUMMY.shape}, got {out.shape}'
print('MLP sanity check passed. Output shape:', out.shape)

MLP sanity check passed. Output shape: torch.Size([4, 3, 6])


### Task 1.2 — RMS Normalization ⚙

Somehow we want to normalise the output from some layers. This is good for deep neural networks. **RMSNorm** used in this lab, normalizes activations by their root-mean-square rather than mean and variance (simpler than LayerNorm). Formula:
$$
a_{\text{norm},i}=\frac{a_i}{\sqrt{\frac{1}{N}\sum_j a_j^2+eps}} \cdot w_i
$$

where $eps$ is a tiny value to avoid division by 0, and the weight $w_i$ is a learnable weight initialised to 1. 

The vector of weights, $\bar{w}$, has the same dimension as the the input vector $\bar{a}$. That means that every dimension of vector $\bar{a}$ will have its own weight. This way, the model is able of amplifying or suppressing each dimension.

PyTorch provides `nn.RMSNorm` (≥2.4), but below is a manual implementation for understanding.

In [8]:
class A2RMSNorm(nn.Module):
    """Root-Mean-Square layer normalization."""
    def __init__(self, config):
        super().__init__()
        self.eps    = config.rms_norm_eps
        self.weight = nn.Parameter(torch.ones(config.hidden_size)) # Initialize trainable parameters this way. Initialized to all ones, so it doesn't change the output at the beginning of training.

    def forward(self, hidden_states):
        """
        1. Compute the variance of the hidden states along the last dimension.
        2. Normalize the hidden states by dividing by the square root of the variance plus epsilon.
        3. Scale the normalized hidden states by the learnable weight parameter.
        """
        rms = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(rms + self.eps)  # rsqrt(x) = 1/sqrt(x)
        return hidden_states * self.weight

In [9]:
# Sanity check: shape is preserved
norm = A2RMSNorm(TEST_CONFIG)
out  = norm(DUMMY)
assert out.shape == DUMMY.shape
print('RMSNorm sanity check passed. Output shape:', out.shape)

RMSNorm sanity check passed. Output shape: torch.Size([4, 3, 6])


### Rotary Position Embeddings (RoPE) — provided

Transformers have no built-in sense of token order. **RoPE** encodes position by *rotating* the query and
key vectors by an angle that depends on their position in the sequence. This lets the attention score
between tokens $i$ and $j$ depend on their *relative* distance $i - j$.

The implementation below is simplified from HuggingFace and is provided — no changes needed.

In [10]:
def rotate_half(x):
    """Rotate half the hidden dims: [x1, x2] → [-x2, x1]."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, rope_rotations, unsqueeze_dim=1):
    """Apply precomputed RoPE (cos, sin) rotations to queries and keys."""
    assert q.shape == k.shape and len(q.shape) == 4
    cos, sin = rope_rotations
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_type, k_type = q.dtype, k.dtype
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed.to(q_type), k_embed.to(k_type)


class A2RotaryEmbedding(nn.Module):
    """Computes RoPE (cos, sin) rotations for a given input sequence."""
    def __init__(self, config, device=None):
        super().__init__()
        head_dim = config.hidden_size // config.num_attention_heads
        dim = int(head_dim * 1.0)  # full rotation (partial_rotary_factor = 1)
        inv_freq = 1.0 / (
            config.rope_theta ** (
                torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim
            )
        )
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    @torch.no_grad()
    def forward(self, x):
        position_ids = torch.arange(0, x.shape[1], device=x.device).unsqueeze(0)
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()
        device_type = x.device.type if x.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, enabled=False):
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()

### Task 1.3 — Multi-Head Attention 🎓

<img src="../../images/mha.svg" width="150" style="background-color: white; padding: 8px;"/>


#### Scaled dot-product attention (causal)

Each token attends to all *past* tokens (including itself) but not future ones — this is **causal masking**.
The lower-triangular mask sets future positions to effectively $-\infty$ before softmax.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_h}} + M\right)V$$

where $M$ is the causal mask and $d_h$ is the per-head dimension.

#### Splitting hidden states into attention heads

The hidden states have shape `(batch_size, seq_length, hidden_size)`. We want to reshape them to `(batch_size, num_attention_heads, seq_length, dim_per_head)` so that we can compute attention for each head separately. 

Remember $\text{dimPerHead} = \frac{\text{num attention heads}}{\text{hidden size}}$.

**Step 1 — `.view(B, N, nr_heads, dim_per_head)`** inserts a head dimension:

```
Before (B=1, N=2, H=4):          After reshape (aka view) → (1, 2, 2, 2):
[                                 [
  [                                 [
    [1, 2, 3, 4],      →               [[1, 2], [3, 4]],   ← position 0
    [5, 6, 7, 8]                       [[5, 6], [7, 8]]    ← position 1
  ]                                 ]
]                                 ]
```

**Step 2 — `.transpose(1, 2)`** swaps the sequence and head axes → `(B, nr_heads, N, dim_per_head)`:

```
After transpose → (1, 2, 2, 2):
[
  [
    [[1, 2], [5, 6]],   ← head 0 sees positions 0 and 1
    [[3, 4], [7, 8]]    ← head 1 sees positions 0 and 1
  ]
]
```

After attention, `.transpose(1, 2).contiguous().view(B, N, H)` merges the heads back.

In [11]:
def scaled_dot_product_attention(q, k, v):
    """Causal scaled dot-product attention. q/k/v: (B, nr_heads, seq_length, dim_per_head)."""
    # NOTE: The attn_mask is a boolean tensor (seq_length, seq_length) used to tell the model which position to pay attention to.
    # Here, we will use it to implement causal masking, which prevents the model from attending
    # to future position in the sequence.
    seq_length = q.shape[-2]
    attn_mask  = torch.ones(seq_length, seq_length, device=q.device).tril().bool() # (seq_length, seq_length), True in the lower triangle and False in the upper triangle
    scale      = q.shape[-1] ** 0.5
    attn_weights = q @ k.transpose(-2, -1) / scale
    attn_weights = torch.softmax(attn_weights + attn_mask, dim=-1)
    output = attn_weights @ v
    return output


class A2Attention(nn.Module):
    """Multi-head attention with RoPE and QK-RMSNorm."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.W_q = nn.Linear(config.hidden_size, config.hidden_size, bias=False) # these are square matrices since they compute y = W * x, y and x same shape (hidden_size,)
        self.W_k = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_v = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_o = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.q_norm = A2RMSNorm(config)  # normalize Q and K before attention
        self.k_norm = A2RMSNorm(config)

    def forward(self, hidden_states, rope_rotations):

        # Divide the hidden states into multiple heads, so normalisation and attention are computed in each head separately.
        B, N, H = hidden_states.shape
        n_h     = self.config.num_attention_heads
        d_h     = H // n_h

        # Normalize queries and keys before splitting into heads 
        q = self.q_norm(self.W_q(hidden_states))  # (B, N, H)
        k = self.k_norm(self.W_k(hidden_states))
        v = self.W_v(hidden_states)

        # (B, N, H) → (B, n_h, N, d_h)
        # First reshape to (batch_size, seq_length, num_attention_heads, dim_per_head) 
        # then transpose to (batch_size, num_attention_heads, seq_length, dim_per_head) with transpose(1, 2)
        q = q.view(B, N, n_h, d_h).transpose(1, 2)
        k = k.view(B, N, n_h, d_h).transpose(1, 2)
        v = v.view(B, N, n_h, d_h).transpose(1, 2)

        if rope_rotations is not None:
            q, k = apply_rotary_pos_emb(q, k, rope_rotations)

        # (B, n_h, N, d_h) → (B, N, H)
        attn_out = scaled_dot_product_attention(q, k, v) # shape (B, n_h, N, d_h)

        # Now we need to reshape this back to (batch_size, seq_length, hidden_size) to pass through the final linear layer W_o.
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, N, H)

        # For this attention computation, we could have simply used PyTorch built-in sdpa function:
        # attn_output = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        # https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html

        return self.W_o(attn_out)

### Task 1.4 — The full Transformer decoder Layer 🎓

<img src="../../images/fullblock.svg" width="150" style="background-color: white; padding: 8px;"/>


A single Transformer decoder block wraps attention + MLP with **pre-norm** and **residual connections**:

```
x  →  RMSNorm  →  Attention  →  + x  →  h
h  →  RMSNorm  →  MLP        →  + h  →  output
```

The residual (`+ x`, `+ h`) lets gradients flow through without vanishing even with many layers.

In [12]:
class A2DecoderLayer(nn.Module):
    """One Transformer decoder block: pre-norm attention + pre-norm MLP with residuals."""
    def __init__(self, config):
        super().__init__()
        self.norm1     = A2RMSNorm(config)
        self.attention = A2Attention(config)
        self.norm2     = A2RMSNorm(config)
        self.mlp       = A2MLP(config)

    def forward(self, hidden_states, rope_rotations):
        h = self.attention(self.norm1(hidden_states), rope_rotations) + hidden_states
        return self.mlp(self.norm2(h)) + h # sum with h for residual

### Task 1.5 — Complete Transformer Stack ⚙

The full model stacks `num_hidden_layers` decoder blocks between an embedding table and an unembedding projection:

```
token IDs  →  Embedding  →  [DecoderLayer] × L  →  RMSNorm  →  Linear (unembedding)  →  logits
```

RoPE rotations are computed once per forward pass and shared across all layers.

> **Note:** Decoder layers must be wrapped in `nn.ModuleList` — a plain Python list would hide the parameters from PyTorch's optimizer.

In [13]:
class A2Transformer(PreTrainedModel):
    """Full Transformer language model (OLMo-2 style)."""
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)
        self.embedding      = nn.Embedding(config.vocab_size, config.hidden_size)
        self.rotary_emb     = A2RotaryEmbedding(config)

        # The ModuleList makes sure your parameters are registered so that they are included  when you compute the gradients.
        self.decoder_layers = nn.ModuleList([A2DecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm           = A2RMSNorm(config)
        self.unembedding    = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.loss_func      = nn.CrossEntropyLoss(ignore_index=-100)

        # Copilot: This method is defined in the PreTrainedModel class and is used to initialize the weights of the model. 
        # It should be called after you have set up all components of the model, so that it can initialize the weights of those components.
        self.post_init()

    def forward(self, input_ids, labels=None):
        rope = self.rotary_emb(input_ids) # pass this to all the transformer decoder layers

        hidden = self.embedding(input_ids)           # (B, N, H)
        for layer in self.decoder_layers:
            hidden = layer(hidden, rope)
        hidden = self.norm(hidden)
        logits = self.unembedding(hidden)            # (B, N, V) i.e. every token gets predictions to all words in vocab

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = self.loss_func(
                shift_logits.view(-1, self.config.vocab_size), # (batch_size * (seq_length - 1), vocab_size)
                shift_labels.view(-1) # (batch_size * (seq_length - 1),)
            )

        return CausalLMOutput(logits=logits, loss=loss)

In [15]:
# Sanity check: forward pass with integer token IDs
model_test = A2Transformer(TEST_CONFIG)
token_ids  = torch.randint(0, TEST_CONFIG.vocab_size, (4, 3))  # (B=4, N=3)
output     = model_test(token_ids)
expected   = (4, 3, TEST_CONFIG.vocab_size)
assert output.logits.shape == expected, f'Expected {expected}, got {output.logits.shape}'
print('Full model sanity check passed. Logits shape:', output.logits.shape)
# the output should be (4, 3, vocab_size), i.e. logits over vocab for each token.


# With labels (check loss is computed)
output_with_loss = model_test(token_ids, labels=token_ids)
print('Loss:', output_with_loss.loss.item())

Full model sanity check passed. Logits shape: torch.Size([4, 3, 10000])
Loss: 9.20566463470459


---
## Part 2 — Training

### Task 2.1 — Train the model ⚙

We reuse `A1Trainer` from Assignment 1. The only changes are:
- Model class is `A2Transformer` instead of `A1RNNModel`
- Config is `A2ModelConfig`

Suggested hyperparameters: `hidden_size=128`, `num_hidden_layers=3`, `num_attention_heads=8`.  
Set `use_subset=True` for a quick smoke-test before a full GPU run.

In [16]:
tokenizer = load_tokenizer()
tokenizer.model_max_length = 256

config = A2ModelConfig(
    vocab_size            = len(tokenizer),
    hidden_size           = 128,
    intermediate_size     = 256,
    num_attention_heads   = 8,
    num_hidden_layers     = 3,
    rope_theta            = 100000.0,
    hidden_act            = 'silu',
    max_position_embeddings = 256,
    rms_norm_eps          = 1e-6,
)
model = A2Transformer(config)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model created. Parameters: {total_params:,}')

Tokenizer loaded. Vocabulary size: 10000.
Model created. Parameters: 3,053,184


In [ ]:
training_args = TrainingArguments(
    optim                       = 'adamw_torch',
    use_cpu                     = False,
    eval_strategy               = 'epoch',
    output_dir                  = './results',
    num_train_epochs            = 10,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate               = 1e-4,
    lr_scheduler_type           = 'linear',
)

if False:
    dataset = get_dataset(use_subset=False)
    trainer = A1Trainer(model, training_args, dataset['train'], dataset['val'], tokenizer)
    trainer.train()

---
## Part 3 — Text Generation

### Task 3.1 — Next-word prediction ⚙

For a given prompt, the model produces logits at every position.
The last position logit gives the probability distribution over the *next* word.
`argmax` selects the single most likely token.

In [17]:
model = A2Transformer.from_pretrained('./results')
model.eval()
tokenizer = load_tokenizer()
tokenizer.model_max_length = 256
print('Model loaded.')

Loading weights: 100%|██████████| 23/23 [00:00<00:00, 12520.31it/s]
A2Transformer LOAD REPORT from: ./results
Key                                                  | Status     | 
-----------------------------------------------------+------------+-
normaliser.weight                                    | UNEXPECTED | 
decoder_layers.{0, 1, 2}.attention.normaliser.weight | UNEXPECTED | 
decoder_layers.{0, 1, 2}.attention.W_o.bias          | UNEXPECTED | 
decoder_layers.{0, 1, 2}.attention.W_q.bias          | UNEXPECTED | 
decoder_layers.{0, 1, 2}.mlp.linear_right.bias       | UNEXPECTED | 
decoder_layers.{0, 1, 2}.mlp.linear_top.bias         | UNEXPECTED | 
decoder_layers.{0, 1, 2}.attention.W_k.bias          | UNEXPECTED | 
decoder_layers.{0, 1, 2}.attention.W_v.bias          | UNEXPECTED | 
decoder_layers.{0, 1, 2}.mlp.linear_left.bias        | UNEXPECTED | 
decoder_layers.{0, 1, 2}.normaliser.weight           | UNEXPECTED | 
decoder_layers.{0, 1, 2}.attention.q_norm.weight     | MISSING

Tokenizer loaded. Vocabulary size: 10000.
Model loaded.


In [18]:
text      = 'She lives in San'
input_ids = tokenizer(text, return_tensors='pt').input_ids
i2t       = get_i2t(tokenizer.vocab)

with torch.no_grad():
    output = model(input_ids)

# Greedy: argmax over the last token's logits
next_id    = torch.argmax(output.logits[0, -1, :]).item()
print(f'Prompt: "{text}"')
print(f'Predicted next word: "{i2t[next_id]}"')

# Top-5 alternatives
top5 = torch.topk(output.logits[0, -1, :], 5)
print('\nTop-5 candidates:')
for score, idx in zip(top5.values, top5.indices):
    print(f'  {i2t[idx.item()]:<20}  logit={score.item():.3f}')

Prompt: "She lives in San"
Predicted next word: "according"

Top-5 candidates:
  according             logit=5.220
  aside                 logit=4.968
  the                   logit=4.909
  although              logit=4.600
  during                logit=4.544


Really bad result...

# TODO: look this sampling generation more into detail

### Task 3.2 — Sampling generation 🎓

Greedy decoding always picks the most likely token, producing repetitive text.
**Sampling** draws from the predicted distribution instead, with two optional controls:

- **Temperature** `T`: divide logits by `T` before softmax.
  - `T > 1` → flatter distribution → more random
  - `T < 1` → sharper distribution → more greedy
- **Top-K**: restrict sampling to the `K` highest-probability tokens,
  setting all others to $-\infty$. Prevents sampling from very unlikely tails.

In [19]:
def generate(model, tokenizer, prompt, max_length=100, temperature=1.0, topk=None):
    """
    Generate text via autoregressive sampling.

    Args:
        model:       language model
        tokenizer:   tokenizer used to encode/decode text
        prompt:      conditioning string
        max_length:  maximum number of new tokens
        temperature: >1 → more random; <1 → more greedy
        topk:        if set, restrict sampling to top-K tokens
    """
    device    = next(model.parameters()).device
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    eos_id    = tokenizer.eos_token_id

    with torch.no_grad():
        for _ in range(max_length):
            logits = model(input_ids).logits[0, -1, :]  # (V,)

            # Temperature scaling
            logits = logits / temperature

            # Top-K truncation: zero out everything outside top-K
            if topk is not None:
                topk_vals, topk_idx = torch.topk(logits, k=topk)
                filtered = torch.full_like(logits, float('-inf'))
                filtered.scatter_(0, topk_idx, topk_vals)
                logits = filtered

            next_id = Categorical(logits=logits).sample()
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)

            if eos_id is not None and next_id.item() == eos_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [20]:
# Generate with the trained model
prompts = [
    'In natural language processing, a Transformer',
    'Is Stockholm the capital of Sweden? The answer is',
    'Write a Python program that reverses a list.',
]

for prompt in prompts:
    print(f'Prompt: {prompt}')
    print('  temp=1.0, topk=None :', generate(model, tokenizer, prompt, max_length=50, temperature=1.0))
    print('  temp=0.7, topk=50   :', generate(model, tokenizer, prompt, max_length=50, temperature=0.7, topk=50))
    print()

Prompt: In natural language processing, a Transformer
  temp=1.0, topk=None : in natural language processing , a at in the the some on `` where the the the the the class= aside angola proportional the `` the the the when `` the `` the when `` for the eisenhower geography cholesterol during `` berkeley monitor during the `` the where geography resin anderson
  temp=0.7, topk=50   : in natural language processing , a while geography where the the the the the the in the the in while belfast openly while belfast treason simplest efficacy geography the in during the the the while refuge flows precisely sites implications parameters santo this oceans counterparts approved

Prompt: Is Stockholm the capital of Sweden? The answer is
  temp=1.0, topk=None : is stockholm the capital of sweden ? the answer is the the the the the the the the while let owner viable smooth proposals instructions the on bahá'ís resigned in boot relies pipeline variance profitable where patch urban owner becomes recept

Real crap.

### Task 3.3 — Compare with OLMo-2 🎓

Load the pre-trained `allenai/OLMo-2-0425-1B` model and generate text for the same prompts.
The generation function works with any HuggingFace `AutoModelForCausalLM` since it only calls `.forward`.

**Optional:** verify that your implementation is correct by copying OLMo-2 weights into `A2Transformer`
and checking that the logits match exactly.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

if False:
    local_dir      = '../../../OLMo-2-0425-1B'
    olmo_tokenizer = AutoTokenizer.from_pretrained(local_dir)
    olmo_model     = AutoModelForCausalLM.from_pretrained(local_dir)
    olmo_model.eval()
    print('OLMo-2 loaded.')

    for prompt in prompts:
        print(f'Prompt: {prompt}')
        print('  OLMo-2 (temp=1.0) :', generate(olmo_model, olmo_tokenizer, prompt, max_length=60, temperature=1.0))
        print()

**OLMo-2 outputs**. Super good.

---

**Prompt:** `In natural language processing, a Transformer`

> In natural language processing, a Transformer is a self-attention mechanism. It can be found in two different types: sequence-to-sequence and transfer learning Transformer. Both the approaches provides better results.
>
> Depending on the type of problem you are trying to solve, one of these methods might be…

---

**Prompt:** `Is Stockholm the capital of Sweden? Answer yes or no. The answer is`

> Yes. I supposed that I have to go there.
>
> How did you/I do? Answer yes or no. I/we went there.
>
> Can anyone help me? Answer yes or no. No. Ok. Thank you.
>
> Are there many tourists in Germany…

---

**Prompt:** `Write a Python program that reverses a list.`

> Unroll each nested list via reverse, then return old_list reversed
>
> ```python
> def reverselist(old_list):
>     '''Original list of lists'''
>     new_list = reverse(old_list, reverse)
>     # Unroll new list by reversing each nested...
> ```